# Chapter 2. Programming Probabilistically

In [ ]:
import os
import warnings

import arviz as az
import matplotlib.pyplot as plt
import pandas as pd

import jax.numpy as jnp
from jax import random


import numpyro
import numpyro.distributions as dist
import numpyro.optim as optim

from numpyro.infer import MCMC, NUTS, Predictive

from numpyro.diagnostics import print_summary

from numpyro.infer import SVI, Trace_ELBO

from numpyro.infer.autoguide import AutoLaplaceApproximation

seed=4321

if "SVG" in os.environ:
    %config InlineBackend.figure_formats = ["svg"]
warnings.formatwarning = lambda message, category, *args, **kwargs: "{}: {}\n".format(
    category.__name__, message
)
az.style.use("arviz-darkgrid")
numpyro.set_platform("cpu") # or "gpu", "tpu" depending on system

## `numpyro` primer

In [ ]:
trials = 4
theta_real = 0.35  # unknown value in a real experiment
# data = stats.bernoulli.rvs(p=theta_real, size=trials)
data = dist.Bernoulli(probs=theta_real).sample(random.PRNGKey(1), (trials,))
data

In [ ]:
def model(data):
    # a priori
    θ = numpyro.sample('θ', dist.Beta(1., 1.))
    # likelihood
    numpyro.sample('y', dist.Bernoulli(probs=θ), obs=data)

kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=1500, num_chains=2)
mcmc.run(random.PRNGKey(1), data=data)

### Summarizing the posterior

In [ ]:
az.plot_trace(az.from_numpyro(mcmc), compact=False)
plt.show()

In [ ]:
mcmc.print_summary()

#### Posterior-based decisions

In [ ]:
az.plot_posterior(az.from_numpyro(mcmc))
plt.show()

In [ ]:
az.plot_posterior(az.from_numpyro(mcmc), rope=[0.45, .55])

In [ ]:
az.plot_posterior(az.from_numpyro(mcmc), ref_val=0.5)

In [ ]:
mcmc.get_samples(group_by_chain=True)

In [ ]:
grid = jnp.linspace(start=0, stop=1, num=200)
θ_pos = mcmc.get_samples()["θ"]
lossf_a = [jnp.mean(abs(i - θ_pos)) for i in grid]
lossf_b = [jnp.mean((i - θ_pos)**2) for i in grid]

for lossf, c in zip([lossf_a, lossf_b], ['C0', 'C1']):
    mini = jnp.argmin(jnp.asarray(lossf))
    plt.plot(grid, lossf, c)
    plt.plot(grid[mini], lossf[mini], 'o', color=c)
    plt.annotate('{:.2f}'.format(grid[mini]),
                 (grid[mini], lossf[mini] + 0.03), color=c)
    plt.yticks([])
    plt.xlabel(r'$\hat \theta$')

In [ ]:
jnp.mean(θ_pos), jnp.median(θ_pos)

In [ ]:
lossf = []
for i in grid:
    if i < 0.5:
        f = jnp.mean(jnp.pi * θ_pos / jnp.abs(i - θ_pos))
    else:
        f = jnp.mean(1 / (i - θ_pos))
    lossf.append(f)

mini = jnp.argmin(jnp.asarray(lossf))
plt.plot(grid, lossf)
plt.plot(grid[mini], lossf[mini], 'o')
plt.annotate('{:.2f}'.format(grid[mini]),
             (grid[mini] + 0.01, lossf[mini] + 0.1))
plt.yticks([])
plt.xlabel(r'$\hat \theta$')

## Gaussian inferences

In [ ]:
data = pd.read_csv('../data/chemical_shifts.csv', header=None)
data.head()

In [ ]:
data = jnp.asarray(data)
data.shape

In [ ]:
az.plot_kde(data, rug=True)
plt.yticks([0], alpha=0)

 <center><img src="../_static/imgs/B11197_02_08.png" width="600"></center>

In [ ]:
def model(obs=None):
    μ = numpyro.sample('μ', dist.Uniform(low=40., high=70.))
    σ = numpyro.sample('σ', dist.HalfNormal(scale=10.))
    
    y = numpyro.sample('y', dist.Normal(loc=μ, scale=σ), obs=obs)

    
kernel = NUTS(model)
mcmc2 = MCMC(kernel, num_warmup=500, num_samples=1000, num_chains=2)
mcmc2.run(random.PRNGKey(seed), obs=data)

In [ ]:
def model(obs=None):
    μ = numpyro.sample('μ', dist.Uniform(low=40., high=70.))
    σ = numpyro.sample('σ', dist.HalfNormal(scale=10.))
    
    y = numpyro.sample('y', dist.Normal(loc=μ, scale=σ), obs=obs)

guide = AutoLaplaceApproximation(model)
svi = SVI(model, guide, optim=optim.Adam(1), loss=Trace_ELBO(), obs=data)
svi_result = svi.run(random.PRNGKey(seed), num_steps=2000)
svi_result.params

In [ ]:
samples = guide.sample_posterior(random.PRNGKey(1), svi_result.params, sample_shape=(1000,))
print_summary(samples, 0.89, False)

In [ ]:
az.summary(samples)

In [ ]:
idata_svi = az.from_dict(posterior={k: v[None, ...] for k, v in samples.items()})
az.plot_pair(idata_svi, var_names=['μ', 'σ'], kind='kde', marginals=True)

In [ ]:
az.plot_trace(az.from_numpyro(mcmc2), compact=False)
plt.show()

In [ ]:
az.plot_pair(az.from_numpyro(mcmc2), var_names=['μ', 'σ'], kind='kde', marginals=True)

In [ ]:
mcmc2.print_summary()

In [ ]:
az.summary(mcmc2)

---

In [ ]:
prior = Predictive(mcmc2.sampler.model, num_samples=10)
prior_p = prior(random.PRNGKey(seed), obs=data)

In [ ]:
pred = Predictive(model=mcmc2.sampler.model, posterior_samples=mcmc2.get_samples(), return_sites=['y'])
post_p = pred(random.PRNGKey(seed))

In [ ]:
# post_p['y'] = post_p['y'].squeeze()
# post_p['y'] = jnp.expand_dims(post_p['y'], axis=1) --> Seems line not needed
post_p['y'].shape

In [ ]:
post_p['y'] = post_p['y'][:50]

In [ ]:
post_p['y'].shape

In [ ]:
jnp.sort(post_p['y'])

In [ ]:
# samples = az.from_numpyro(mcmc2, prior=prior_p, posterior_predictive=post_p)
samples = az.from_numpyro(mcmc2, prior=prior_p, posterior_predictive=post_p) # Priop p seems not required.
# az.summary(samples)

In [ ]:
samples.groups()

In [ ]:
fig, ax = plt.subplots()
az.plot_kde(post_p['y'], ax=ax, plot_kwargs={'alpha': 0.5})

In [ ]:
az.plot_ppc(samples, mean=True, observed=True)
plt.xlim(40, 70)

---

### Robust inferences

In [ ]:
plt.figure(figsize=(10, 6))
x_values = jnp.linspace(start=-10, stop=10, num=500)
for df in [1, 2, 30]:
    distri = dist.StudentT(df)
    x_pdf = jnp.exp(distri.log_prob(x_values))
    plt.plot(x_values, x_pdf, label=fr'$\nu = {df}$', lw=3)

x_pdf = jnp.exp(dist.Normal().log_prob(x_values))
plt.plot(x_values, x_pdf, 'k--', label=r'$\nu = \infty$')
plt.xlabel('x')
plt.yticks([])
plt.legend()
plt.xlim(-5, 5)

 <center><img src="../_static/imgs/B11197_02_13.png" width="600"></center>

In [ ]:
def model(obs=None):
    μ = numpyro.sample('μ', dist.Uniform(low=40., high=75.))
    σ = numpyro.sample('σ', dist.HalfNormal(scale=10.))
    ν = numpyro.sample('ν', dist.Exponential(rate=1/30))
    
    y = numpyro.sample('y', dist.StudentT(ν, loc=μ, scale=σ), obs=obs)

    
kernel = NUTS(model)
mcmc3 = MCMC(kernel, num_warmup=500, num_samples=500, num_chains=2)
mcmc3.run(random.PRNGKey(seed), obs=data)

In [ ]:
az.plot_trace(az.from_numpyro(mcmc3), compact=False)
plt.show()

In [ ]:
az.summary(mcmc3)

In [ ]:
prior = Predictive(mcmc3.sampler.model, num_samples=10)
prior_p = prior(random.PRNGKey(seed), obs=data)

pred = Predictive(model=mcmc3.sampler.model, posterior_samples=mcmc3.get_samples(), return_sites=['y'])
post_p = pred(random.PRNGKey(seed))

In [ ]:
post_p['y'] = post_p['y'][:100]

In [ ]:
samples = az.from_numpyro(mcmc3, prior=prior_p, posterior_predictive=post_p) ## CHECK THIS

In [ ]:
az.plot_ppc(samples, mean=True, observed=True, color='C0')
plt.xlim(40, 70)

# Tips example

In [ ]:
tips = pd.read_csv('../data/tips.csv')
tips.tail()

In [ ]:
import seaborn as sns
sns.violinplot(x='day', y='tip', data=tips)

In [ ]:
tip = tips['tip'].values
idx = pd.Categorical(tips['day'],
                     categories=['Thur', 'Fri', 'Sat', 'Sun']).codes
groups = len(jnp.unique(idx))

In [ ]:
def model(N=len(idx), obs=None):
    μ = numpyro.sample('μ', dist.Normal(loc=0., scale=10.), sample_shape=(groups,))
    σ = numpyro.sample('σ', dist.HalfNormal(scale=10.), sample_shape=(groups,))
    
    with numpyro.plate("N", N):
        numpyro.sample('y', dist.Normal(loc=μ[idx], scale=σ[idx]), obs=obs)

    
kernel = NUTS(model)
mcmc4 = MCMC(kernel, num_warmup=1000, num_samples=4000, num_chains=2)
mcmc4.run(random.PRNGKey(seed), obs=tip)

In [ ]:
az.plot_trace(az.from_numpyro(mcmc4), compact=False)
plt.show()

In [ ]:
distri = dist.Normal()

_, ax = plt.subplots(3, 2, figsize=(14, 8), constrained_layout=True)

comparisons = [(i, j) for i in range(4) for j in range(i+1, 4)]
pos = [(k, l) for k in range(3) for l in (0, 1)]

for (i, j), (k, l) in zip(comparisons, pos):
    means_diff = mcmc4.get_samples()['μ'][:, i] - mcmc4.get_samples()['μ'][:, j]
    d_cohen = (means_diff / jnp.sqrt((mcmc4.get_samples()['σ'][:, i]**2 + mcmc4.get_samples()['σ'][:, j]**2) / 2)).mean()
    ps = distri.cdf(d_cohen/(2**0.5))
#     import pdb;pdb.set_trace()
    means_diff = jnp.asarray(means_diff)
    az.plot_posterior(means_diff.copy(), ref_val=0, ax=ax[k, l])
    ax[k, l].set_title(f'$\mu_{i}-\mu_{j}$')
    ax[k, l].plot(
        0, label=f"Cohen's d = {d_cohen:.2f}\nProb sup = {ps:.2f}", alpha=0)
    ax[k, l].legend()

# Hierarchical Models

 <center><img src="../_static/imgs/B11197_02_19.png" width="600"></center>

In [ ]:
N_samples = [30, 30, 30]
G_samples = [18, 18, 18]  # [3, 3, 3]  [18, 3, 3]

In [ ]:
N_samples[0]

In [ ]:
group_idx = jnp.repeat(jnp.arange(len(N_samples)), N_samples[0])
data = []

In [ ]:
for i in range(0, len(N_samples)):
    data.extend(jnp.repeat(jnp.asarray([1, 0]), jnp.asarray([G_samples[i], N_samples[i]-G_samples[i]])))

In [ ]:
data = jnp.asarray(data)

In [ ]:
def model(obs=None):
    μ = numpyro.sample('μ', dist.Beta(1.,1.))
    κ = numpyro.sample('κ', dist.HalfNormal(scale=10.))
    θ = numpyro.sample('θ', dist.Beta(μ*κ, (1.0-μ)*κ), sample_shape=(len(N_samples),))
    
#     with numpyro.plate("N", N):
    numpyro.sample('y', dist.Bernoulli(probs=θ[group_idx]), obs=obs, sample_shape=(len(N_samples),))

    
kernel = NUTS(model)
mcmc5 = MCMC(kernel, num_warmup=1000, num_samples=2000, num_chains=2)
mcmc5.run(random.PRNGKey(seed), obs=data.copy())  # .copy() needed since data in list above

In [ ]:
az.plot_trace(az.from_numpyro(mcmc5), compact=False)
plt.show()

In [ ]:
az.summary(mcmc5)

In [ ]:
prior = Predictive(mcmc5.sampler.model, num_samples=10)
prior_p = prior(random.PRNGKey(seed), obs=data)

pred = Predictive(model=mcmc5.sampler.model, posterior_samples=mcmc5.get_samples(), return_sites=['y'])
post_p = pred(random.PRNGKey(seed))

samples = az.from_numpyro(mcmc5, prior=prior_p, posterior_predictive=post_p)
az.plot_ppc(samples, mean=True, observed=True, color='C0')

In [ ]:
len(mcmc5.get_samples()['μ'])

In [ ]:
x = jnp.linspace(start=0, stop=1, num=100)
for i in random.randint(random.PRNGKey(1), shape=(100,), minval=0, maxval=len(mcmc5.get_samples()['μ'])):
    u = mcmc5.get_samples()['μ'][i]
    k = mcmc5.get_samples()['κ'][i]
    pdf = jnp.exp(dist.Beta(u*k, (1.0-u)*k).log_prob(x))
    plt.plot(x, pdf,  'C1', alpha=0.2)

u_mean = mcmc5.get_samples()['μ'].mean()
k_mean = mcmc5.get_samples()['κ'].mean()
                  
distri = dist.Beta(u_mean*k_mean, (1.0-u_mean)*k_mean)
pdf = jnp.exp(distri.log_prob(x))
mode = x[jnp.argmax(pdf)]
mean = distri.mean
plt.plot(x, pdf, lw=3, label=f'mode = {mode:.2f}\nmean = {mean:.2f}')
plt.yticks([])

plt.legend()
plt.xlabel('$θ_{prior}$')
plt.tight_layout()

In [ ]:
cs_data = pd.read_csv('../data/chemical_shifts_theo_exp.csv')
diff = cs_data.theo.values - cs_data.exp.values
idx = pd.Categorical(cs_data['aa']).codes
groups = len(jnp.unique(idx))

In [ ]:
cs_data.head()

In [ ]:
def model(obs=None):
    μ = numpyro.sample('μ', dist.Normal(loc=0., scale=10.), sample_shape=(groups,))
    σ = numpyro.sample('σ', dist.HalfNormal(scale=10.), sample_shape=(groups,))
    
    numpyro.sample('y', dist.Normal(loc=μ[idx], scale=σ[idx]), obs=obs, sample_shape=(len(cs_data),))

    
kernel = NUTS(model)
mcmc6 = MCMC(kernel, num_warmup=500, num_samples=500, num_chains=2)
mcmc6.run(random.PRNGKey(seed), obs=diff)

In [ ]:
def model(obs=None):
    # hyperpriors
    μ_μ = numpyro.sample('μ_μ', dist.Normal(loc=0., scale=10.))
    σ_μ = numpyro.sample('σ_μ', dist.HalfNormal(scale=10.))
    
    # priors
    μ = numpyro.sample('μ', dist.Normal(loc=μ_μ, scale=σ_μ), sample_shape=(groups,))
    σ = numpyro.sample('σ', dist.HalfNormal(scale=10.), sample_shape=(groups,))
    
    numpyro.sample('y', dist.Normal(loc=μ[idx], scale=σ[idx]), obs=obs, sample_shape=(len(cs_data),))

    
kernel = NUTS(model)
mcmc7 = MCMC(kernel, num_warmup=500, num_samples=500, num_chains=2)
mcmc7.run(random.PRNGKey(seed), obs=diff)

In [ ]:
axes = az.plot_forest([mcmc6, mcmc7],
                         model_names=['n_h', 'h'],
                         var_names='μ', combined=False, colors='cycle')
y_lims = axes[0].get_ylim()
axes[0].vlines(jnp.mean(mcmc7.get_samples()['μ_μ']), color='k', *y_lims)